In [1]:
import sys
import os

sys.path.append(f"{os.environ['MDS']}/code")
from prometheus_evaluator import PrometheusEvaluator, INSTRUCTION, RUBRICS
from project_utils import load_generations


Setting up Prometheus

In [5]:
score_descriptions = {
    "score1_description": "Very poor",
    "score2_description": "Poor",
    "score3_description": "Barely acceptable",
    "score4_description": "Good",
    "score5_description": "Very Good"
}
summeval_criteria = {
    "Relevance": {"criteria": "How well does the summary capture the key points of the documents? Consider whether all and only the important aspects are contained in the summary."},
    "Consistency": {"criteria": "How consistent are the facts in the summary with the facts in the original documents? Consider whether the summary does reproduce all facts accurately and does not make up untrue information. Do not use your own knowledge, and consider the documents as truth."},
    "Fluency": {"criteria": "What is the quality of individual sentences, are they well-written and grammatically correct? Consider the quality of individual sentences."},
    "Coherence": {"criteria": "What is the quality of all sentences collectively? How well do they fit together, and do they sound natural? Consider the quality of the summary as a whole."}
}

summeval_rubric = {}
for key, crit in summeval_criteria.items():
    crit.update(score_descriptions)
    crit["criteria"] += f" Only evaluate the summary for {key.lower()}. Do not penalize the summary for issues that are outside the scope of {key.lower()}."
    summeval_rubric[key] = crit

input_data = "/home/adeluci2/projects/multidoc_summarization/data/multinews_plus/test.jsonl"
output_data = "/home/adeluci2/projects/multidoc_summarization/results_mayarr/SummN-MultiXScience/multinews_plus/test/generated_with_ids.jsonl"

# 1. Load the data
dataset = {"source": [], "predictions": [], "reference": []}
for i, example in enumerate(load_generations(input_data, output_data)):
    dataset["source"].append(example["source"])
    dataset["predictions"].append(example["generated"])
    dataset["reference"].append(example["target"])
    if i > 1:
        break

# 2. Initialize evaluator
evaluator = PrometheusEvaluator(
    api_base="http://icgpu05:22659/v1",
)

instruction = []
for source_docs in dataset["source"]:
    docs = "\n\nDocument: ".join(source_docs)
    docs = "Document: " + docs
    ins = INSTRUCTION.format(source=docs)
    instruction.append(ins)


In [6]:
for key, crit in summeval_rubric.items():
    print(f"{key}:\n{crit}")


Relevance:
{'criteria': 'How well does the summary capture the key points of the documents? Consider whether all and only the important aspects are contained in the summary. Only evaluate the summary for relevance. Do not penalize the summary for issues that are outside the scope of relevance.', 'score1_description': 'Very poor', 'score2_description': 'Poor', 'score3_description': 'Barely acceptable', 'score4_description': 'Good', 'score5_description': 'Very Good'}
Consistency:
{'criteria': 'How consistent are the facts in the summary with the facts in the original documents? Consider whether the summary does reproduce all facts accurately and does not make up untrue information. Do not use your own knowledge, and consider the documents as truth. Only evaluate the summary for consistency. Do not penalize the summary for issues that are outside the scope of consistency.', 'score1_description': 'Very poor', 'score2_description': 'Poor', 'score3_description': 'Barely acceptable', 'score4_

In [3]:
for dimension, rubric in summeval_rubric.items():
#     results = evaluator.relative_grade(
#         generation_A=dataset["predictions"],
#         generation_B=dataset["reference"],
#         instruction=instruction,
#         rubric_data=rubric
#     )
    results = evaluator.absolute_grade(
        generation=dataset["reference"],
        instruction=instruction,
        rubric_data=rubric,
        #reference_generation=dataset["reference"],
    )
    print(f"DIMENSION: {dimension}")
    for i, r in enumerate(results):
        print(f"SUMMARY: {dataset['reference'][i]}")
        print(f"SCORE: {r['score']}. {r['feedback']}")
        print("-------")
    print("********************\n")

  0% 0/1 [00:00<?, ?it/s]

04/26/2025 22:18:35 - INFO - HTTP Request: POST http://icgpu05:22659/v1/chat/completions "HTTP/1.1 200 OK"
04/26/2025 22:18:36 - INFO - HTTP Request: POST http://icgpu05:22659/v1/chat/completions "HTTP/1.1 200 OK"
04/26/2025 22:18:37 - INFO - HTTP Request: POST http://icgpu05:22659/v1/chat/completions "HTTP/1.1 200 OK"


100% 1/1 [00:19<00:00, 19.57s/it]


DIMENSION: Relevance
SUMMARY: Wal-mart is selling a book for kids and parents that makes the case that homosexuals can overcome their " sin " and revert to heterosexuality, q salt lake reports. Chased by an elephant, the gospel truth about today's stampeding sexuality is by janice barrett graham — the wife of stephen graham, who leads the standard of liberty group and its philosophy of " pray the gay away. " graham says chasing an elephant's goal is to " shed the clear light of truth on today's dark and tangled ideas about male and female, proper gender roles, the law of chastity, and the god-given sexual appetite. " it includes the story of her own son, who claims to have turned from gay to straight after seeing the error of his ways. The book is hardly a bestseller, and for the folks over at queerty, the fact that wal-mart is lending its titanic reach to an anti-gay book is just another example of the big-box store's bias.
SCORE: 2. The response does capture some of the key points of

  0% 0/1 [00:00<?, ?it/s]

...

DIMENSION: Fluency
SUMMARY: Wal-mart is selling a book for kids and parents that makes the case that homosexuals can overcome their " sin " and revert to heterosexuality, q salt lake reports. Chased by an elephant, the gospel truth about today's stampeding sexuality is by janice barrett graham — the wife of stephen graham, who leads the standard of liberty group and its philosophy of " pray the gay away. " graham says chasing an elephant's goal is to " shed the clear light of truth on today's dark and tangled ideas about male and female, proper gender roles, the law of chastity, and the god-given sexual appetite. " it includes the story of her own son, who claims to have turned from gay to straight after seeing the error of his ways. The book is hardly a bestseller, and for the folks over at queerty, the fact that wal-mart is lending its titanic reach to an anti-gay book is just another example of the big-box store's bias.
SCORE: 2. The response does capture the essence of both documen

  0% 0/1 [00:00<?, ?it/s]

04/26/2025 22:19:36 - INFO - HTTP Request: POST http://icgpu05:22659/v1/chat/completions "HTTP/1.1 200 OK"
04/26/2025 22:19:37 - INFO - HTTP Request: POST http://icgpu05:22659/v1/chat/completions "HTTP/1.1 200 OK"
04/26/2025 22:19:37 - INFO - HTTP Request: POST http://icgpu05:22659/v1/chat/completions "HTTP/1.1 200 OK"


100% 1/1 [00:20<00:00, 20.39s/it]

DIMENSION: Coherence
SUMMARY: Wal-mart is selling a book for kids and parents that makes the case that homosexuals can overcome their " sin " and revert to heterosexuality, q salt lake reports. Chased by an elephant, the gospel truth about today's stampeding sexuality is by janice barrett graham — the wife of stephen graham, who leads the standard of liberty group and its philosophy of " pray the gay away. " graham says chasing an elephant's goal is to " shed the clear light of truth on today's dark and tangled ideas about male and female, proper gender roles, the law of chastity, and the god-given sexual appetite. " it includes the story of her own son, who claims to have turned from gay to straight after seeing the error of his ways. The book is hardly a bestseller, and for the folks over at queerty, the fact that wal-mart is lending its titanic reach to an anti-gay book is just another example of the big-box store's bias.
SCORE: 3. The response does capture some of the essence and d